
# 🎯 NB06 — Marketing Intelligence Layer
## Investor Intelligence Platform — FIIs Brasil 🇧🇷
### CRISP-DM Phase: Evaluation

| | |
|---|---|
| **Inputs** | `silver_enriched.parquet` · `articles_sentiment.parquet` · `bm25_index.pkl` · `tfidf_vectorizer.pkl` · `document_relevance.parquet` |
| **Outputs** | `data/gold/marketing_intelligence/` — `mi_signals.parquet` · `mi_top_articles.parquet` · `mi_funnel.parquet` |
| **Engines** | BM25 + TF-IDF + Sentiment + PySpark aggregations |

## 🎯 O que este notebook faz

1. **Entidades FII** — catalogo de 15 FIIs líquidos para análise.
2. **Funil TOFU/MOFU/BOFU** — classifica documentos por estágio de intenção.
3. **MI Signals** — por FII: `sentiment_avg`, `mentions`, `risk_score`, `opportunity_score`, `relevance_bm25`.
4. **MI Top Articles** — top 10 artigos por FII rankeados por score híbrido × sentimento.
5. **MI Funnel** — volume por estágio (topo, meio, fundo) por fonte.

## 📐 Pipeline de Score MI

```
Para cada FII entity:
  1. BM25.get_scores(fii_query)       → relevance_bm25
  2. TF-IDF cosine_similarity()       → relevance_tfidf
  3. score_hybrid = 0.4×TF + 0.6×BM  (MinMax norm)
  4. Juntar sentimento (silver_enriched)
  5. mi_score = 0.5×relevance + 0.3×|polarity| + 0.2×(flag_dividendo + flag_oportunidade)
```

## 📦 Seção 1 — Imports

In [1]:

import sys, os, pickle, warnings
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
import re, random, unicodedata, json
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from rank_bm25 import BM25Okapi

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pyspark

warnings.filterwarnings("ignore")
print(f"Python    : {sys.version.split()[0]}")
print(f"PySpark   : {pyspark.__version__}")

# rank_bm25 doesn't expose __version__, so we handle it gracefully
try:
    bm25_version = __import__('rank_bm25').__version__
except AttributeError:
    bm25_version = "unknown"
print(f"rank_bm25 : {bm25_version}")


Python    : 3.12.12
PySpark   : 4.1.2
rank_bm25 : unknown


## ⚙️ Seção 2 — Configuração e SparkSession

In [2]:

PROJECT_ROOT = Path(os.getcwd()).resolve()
_lim = 6
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT.parent != PROJECT_ROOT and _lim > 0:
    PROJECT_ROOT = PROJECT_ROOT.parent; _lim -= 1
sys.path.insert(0, str(PROJECT_ROOT))

from config.settings import (
    RANDOM_SEED, SPARK_DRIVER_MEMORY, SPARK_APP_NAME, SPARK_SHUFFLE_PARTS,
)
from src.utils.logger import ingestion_logger, log_quality_check, log_spark_start

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

SILVER_DIR   = PROJECT_ROOT / "data" / "silver"
GOLD_IDX_DIR = PROJECT_ROOT / "data" / "gold" / "tfidf_bm25"
GOLD_SENT_DIR = PROJECT_ROOT / "data" / "gold" / "sentiment"
GOLD_MI_DIR  = PROJECT_ROOT / "data" / "gold" / "marketing_intelligence"
GOLD_MI_DIR.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName(f"{SPARK_APP_NAME}_mi")
    .master("local[*]")
    .config("spark.driver.memory",          SPARK_DRIVER_MEMORY)
    .config("spark.sql.shuffle.partitions", SPARK_SHUFFLE_PARTS)
    .config("spark.sql.adaptive.enabled",   "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

logger = ingestion_logger()
# FIXED: log_spark_start requer 3 argumentos (logger, app_name, spark_version)
log_spark_start(logger, spark.sparkContext.appName, spark.version)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"GOLD_MI_DIR  : {GOLD_MI_DIR}")
print(f"Spark        : {spark.sparkContext.appName} | {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/14 15:51:59 WARN Utils: Your hostname, MacBook-Air-100.local, resolves to a loopback address: 127.0.0.1; using 192.168.15.4 instead (on interface en0)
26/06/14 15:51:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/14 15:52:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/14 15:52:01 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/14 15:52:01 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/14 15:52:01 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/06/14 15:52:01 WARN 

2026-06-14T15:52:03 | INFO     | fii_pipeline.ingestion | SPARK_START | app=FIIIntelligencePlatform_mi | spark=4.1.2
PROJECT_ROOT : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform
GOLD_MI_DIR  : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/data/gold/marketing_intelligence
Spark        : FIIIntelligencePlatform_mi | 4.1.2


## 📂 Seção 3 — Carregar Inputs

In [3]:

# ── Silver Enriched ──────────────────────────────────────────────────────────
def _load_parquet_resilient(paths: List, label: str) -> pd.DataFrame:
    for p in paths:
        if Path(p).exists():
            df = pd.read_parquet(p)
            print(f"  ✅ {label}: {Path(p).name} — {len(df):,} registros")
            return df
    raise FileNotFoundError(f"{label} não encontrado. Caminhos tentados: {paths}")

df_silver = _load_parquet_resilient([
    SILVER_DIR / "silver_enriched.parquet",
    SILVER_DIR / "silver_articles.parquet",
], "Silver Enriched")

df_sent = _load_parquet_resilient([
    GOLD_SENT_DIR / "articles_sentiment.parquet",
], "Articles Sentiment")

# ── BM25 + TF-IDF índices ────────────────────────────────────────────────────
def _load_pkl(path, label):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"{label} não encontrado: {p}\nExecute NB04 primeiro.")
    with open(p, "rb") as f:
        obj = pickle.load(f)
    print(f"  ✅ {label}: {p.name}")
    return obj

bm25_index       = _load_pkl(GOLD_IDX_DIR / "bm25_index.pkl",       "BM25 Index")
tfidf_vectorizer = _load_pkl(GOLD_IDX_DIR / "tfidf_vectorizer.pkl",  "TF-IDF Vectorizer")
corpus_tokens    = _load_pkl(GOLD_IDX_DIR / "corpus_tokens.pkl",     "Corpus Tokens")
tfidf_matrix     = sp.load_npz(str(GOLD_IDX_DIR / "tfidf_matrix.npz"))
doc_map          = pd.read_parquet(GOLD_IDX_DIR / "doc_index_map.parquet")

print(f"\n  TF-IDF matrix  : {tfidf_matrix.shape}")
print(f"  BM25 corpus    : {len(corpus_tokens):,} docs")
print(f"  Doc map        : {len(doc_map):,} entradas")

  ✅ Silver Enriched: silver_enriched.parquet — 126 registros
  ✅ Articles Sentiment: articles_sentiment.parquet — 126 registros
  ✅ BM25 Index: bm25_index.pkl
  ✅ TF-IDF Vectorizer: tfidf_vectorizer.pkl
  ✅ Corpus Tokens: corpus_tokens.pkl

  TF-IDF matrix  : (126, 9449)
  BM25 corpus    : 126 docs
  Doc map        : 126 entradas



## 🏢 Seção 4 — Catálogo de Entidades FII

**15 FIIs líquidos** selecionados para análise de marketing intelligence.
Cada entidade tem:
- `ticker` — código oficial B3
- `full_name` — nome completo
- `segment` — tipo de ativo
- `query_terms` — termos de busca para BM25/TF-IDF

In [4]:

FII_ENTITIES: List[Dict] = [
    {"ticker": "HGLG11", "full_name": "CSHG Logística",     "segment": "logistica",
     "query_terms": ["HGLG11","CSHG logistica","galpao logistico","ativo logistico"]},
    {"ticker": "KNRI11", "full_name": "Kinea Renda Imobiliária","segment": "diversificado",
     "query_terms": ["KNRI11","Kinea renda","diversificado","laje corporativa"]},
    {"ticker": "XPML11", "full_name": "XP Malls",           "segment": "shopping",
     "query_terms": ["XPML11","XP Malls","shopping","varejo FII"]},
    {"ticker": "CSHG11", "full_name": "CSHG Real Estate",   "segment": "laje",
     "query_terms": ["CSHG11","CSHG real estate","laje corporativa","escritorio"]},
    {"ticker": "VISC11", "full_name": "Vinci Shopping Centers","segment": "shopping",
     "query_terms": ["VISC11","Vinci shopping","shopping center","varejo"]},
    {"ticker": "BRCO11", "full_name": "Bresco Logística",   "segment": "logistica",
     "query_terms": ["BRCO11","Bresco logistica","galpao","condominio logistico"]},
    {"ticker": "BTLG11", "full_name": "BTG Pactual Logística","segment": "logistica",
     "query_terms": ["BTLG11","BTG logistica","last mile","logistica urbana"]},
    {"ticker": "MXRF11", "full_name": "Maxi Renda",         "segment": "papel",
     "query_terms": ["MXRF11","Maxi renda","CRI","CRA","papel FII"]},
    {"ticker": "HCRI11", "full_name": "HC Investimentos",   "segment": "hibrido",
     "query_terms": ["HCRI11","HC investimentos","hibrido FII","diversificado"]},
    {"ticker": "ALZR11", "full_name": "Alianza Trust",      "segment": "logistica",
     "query_terms": ["ALZR11","Alianza","ativos logisticos","e-commerce"]},
    {"ticker": "BRCR11", "full_name": "FII BC Fund",        "segment": "laje",
     "query_terms": ["BRCR11","BC Fund","laje AAA","escritorio premium"]},
    {"ticker": "HSML11", "full_name": "HSI Malls",          "segment": "shopping",
     "query_terms": ["HSML11","HSI Malls","shopping regional","varejo"]},
    {"ticker": "VGIP11", "full_name": "Valora CRI",         "segment": "papel",
     "query_terms": ["VGIP11","Valora CRI","credito imobiliario","CRI alto yield"]},
    {"ticker": "GGRC11", "full_name": "GGR Covepi Renda",   "segment": "logistica",
     "query_terms": ["GGRC11","GGR Covepi","built-to-suit","galpao premium"]},
    {"ticker": "RBVA11", "full_name": "Rio Bravo Renda Varejo","segment": "varejo",
     "query_terms": ["RBVA11","Rio Bravo varejo","agencias bancarias","varejo nao-alimentar"]},
]

FUNIL_QUERIES: Dict[str, List[str]] = {
    "TOFU": [
        "o que e fundo imobiliario FII",
        "como investir FII iniciante",
        "FII dividendo renda passiva",
        "melhores FII 2024 2025",
        "FII vs tesouro direto",
    ],
    "MOFU": [
        "analise fundamentalista FII tijolo",
        "comparacao FII logistica shopping laje",
        "vacancia vs inadimplencia FII",
        "yield FII vs inflacao IPCA",
        "como avaliar gestora FII",
    ],
    "BOFU": [
        "comprar HGLG11 agora vale pena",
        "preco cota FII desconto pvp",
        "FII defensivo risco baixo dividendo alto",
        "rebalancear carteira FII vender comprar",
        "FII KNRI11 XPML11 recomendacao",
    ],
}

print(f"✅ Catálogo de FIIs: {len(FII_ENTITIES)} entidades")
for e in FII_ENTITIES:
    print(f"   {e['ticker']:<8} — {e['full_name']:<28} [{e['segment']}]")
print(f"\n✅ Funil: {sum(len(v) for v in FUNIL_QUERIES.values())} queries em {len(FUNIL_QUERIES)} estágios")

✅ Catálogo de FIIs: 15 entidades
   HGLG11   — CSHG Logística               [logistica]
   KNRI11   — Kinea Renda Imobiliária      [diversificado]
   XPML11   — XP Malls                     [shopping]
   CSHG11   — CSHG Real Estate             [laje]
   VISC11   — Vinci Shopping Centers       [shopping]
   BRCO11   — Bresco Logística             [logistica]
   BTLG11   — BTG Pactual Logística        [logistica]
   MXRF11   — Maxi Renda                   [papel]
   HCRI11   — HC Investimentos             [hibrido]
   ALZR11   — Alianza Trust                [logistica]
   BRCR11   — FII BC Fund                  [laje]
   HSML11   — HSI Malls                    [shopping]
   VGIP11   — Valora CRI                   [papel]
   GGRC11   — GGR Covepi Renda             [logistica]
   RBVA11   — Rio Bravo Renda Varejo       [varejo]

✅ Funil: 15 queries em 3 estágios


## 🔍 Seção 5 — Funções de Busca MI

In [5]:

_TOKEN_RE = re.compile(r"[a-zà-ü]{3,}", re.IGNORECASE | re.UNICODE)

def _norm(s: str) -> str:
    return unicodedata.normalize("NFD", s).encode("ascii","ignore").decode("ascii").lower()

def _tok(text: str) -> List[str]:
    if not text: return []
    return [_norm(t) for t in _TOKEN_RE.findall(text.lower())]

def _minmax(arr: np.ndarray) -> np.ndarray:
    mn, mx = arr.min(), arr.max()
    return np.zeros_like(arr) if mx == mn else (arr - mn) / (mx - mn)

def query_relevance(query: str, top_k: int = 50, alpha: float = 0.4) -> pd.DataFrame:
    """Retorna top_k docs com scores TF-IDF, BM25 e híbrido para uma query."""
    q_toks = _tok(query)
    q_str  = " ".join(q_toks)

    s_bm = np.array(bm25_index.get_scores(q_toks)) if q_toks else np.zeros(len(doc_map))
    s_tf = cosine_similarity(
        tfidf_vectorizer.transform([q_str]) if q_str.strip() else
        np.zeros((1, tfidf_matrix.shape[1])), tfidf_matrix
    ).flatten()

    s_bm_n = _minmax(s_bm)
    s_tf_n = _minmax(s_tf)
    s_hyb  = alpha * s_tf_n + (1 - alpha) * s_bm_n

    top_idx = s_hyb.argsort()[::-1][:top_k]
    return pd.DataFrame({
        "doc_index":     top_idx,
        "article_id":    doc_map.iloc[top_idx]["article_id"].values,
        "score_bm25":    s_bm[top_idx].round(4),
        "score_tfidf":   s_tf[top_idx].round(4),
        "score_hybrid":  s_hyb[top_idx].round(4),
    })

print("✅ query_relevance() definida")

✅ query_relevance() definida


## 🧠 Seção 6 — MI Signals: Métricas por FII

In [6]:

print("\n🧠 Calculando MI Signals por FII...")

# Garantir colunas de sentimento em df_silver
_sent_cols = ["polarity_score","sentiment_label","flag_dividendo",
              "flag_oportunidade","flag_risco","flag_crise","flag_vacancia",
              "flag_inadimplencia","n_pos_terms","n_neg_terms"]
for c in _sent_cols:
    if c not in df_silver.columns:
        if c in df_sent.columns:
            df_silver = df_silver.merge(df_sent[["article_id",c]], on="article_id", how="left")
        else:
            df_silver[c] = np.nan if "score" in c or "polarity" in c else False

mi_signals_rows = []

for entity in FII_ENTITIES:
    ticker = entity["ticker"]
    # Combinar relevância de todos os query_terms do ticker
    rel_dfs = [query_relevance(q, top_k=50) for q in entity["query_terms"]]
    df_rel  = pd.concat(rel_dfs, ignore_index=True)
    df_rel  = df_rel.groupby("article_id").agg(
        score_bm25   = ("score_bm25",  "max"),
        score_tfidf  = ("score_tfidf", "max"),
        score_hybrid = ("score_hybrid","max"),
    ).reset_index()

    # Juntar com sentimento
    df_mi = df_rel.merge(
        df_silver[["article_id","polarity_score","sentiment_label",
                   "flag_dividendo","flag_oportunidade","flag_risco",
                   "flag_crise","flag_vacancia","flag_inadimplencia",
                   "source","title"]].drop_duplicates("article_id"),
        on="article_id", how="left"
    )

    n_docs          = len(df_mi)
    avg_polarity    = float(df_mi["polarity_score"].mean()) if n_docs else 0.0
    avg_relevance   = float(df_mi["score_hybrid"].mean())   if n_docs else 0.0
    n_pos           = int((df_mi["sentiment_label"] == "positivo").sum())
    n_neg           = int((df_mi["sentiment_label"] == "negativo").sum())
    n_dividendo     = int(df_mi["flag_dividendo"].fillna(False).sum())
    n_oportunidade  = int(df_mi["flag_oportunidade"].fillna(False).sum())
    n_risco         = int(df_mi["flag_risco"].fillna(False).sum())
    n_crise         = int(df_mi["flag_crise"].fillna(False).sum())
    n_vacancia      = int(df_mi["flag_vacancia"].fillna(False).sum())
    n_inadimplencia = int(df_mi["flag_inadimplencia"].fillna(False).sum())

    # MI Score composto
    risk_score = round((n_risco + 2*n_crise + 2*n_vacancia + 2*n_inadimplencia) / max(n_docs, 1), 4)
    opp_score  = round((n_dividendo + n_oportunidade) / max(n_docs, 1), 4)

    mi_signals_rows.append({
        "ticker":           ticker,
        "full_name":        entity["full_name"],
        "segment":          entity["segment"],
        "mentions":         n_docs,
        "sentiment_avg":    round(avg_polarity, 4),
        "relevance_avg":    round(avg_relevance, 4),
        "pct_positivo":     round(n_pos / max(n_docs, 1) * 100, 2),
        "pct_negativo":     round(n_neg / max(n_docs, 1) * 100, 2),
        "n_dividendo":      n_dividendo,
        "n_oportunidade":   n_oportunidade,
        "n_risco":          n_risco,
        "n_crise":          n_crise,
        "n_vacancia":       n_vacancia,
        "n_inadimplencia":  n_inadimplencia,
        "risk_score":       risk_score,
        "opportunity_score":opp_score,
        "mi_score":         round(0.5*avg_relevance + 0.3*abs(avg_polarity) + 0.2*opp_score, 4),
    })
    print(f"  {ticker:<8} → {n_docs:>4} docs | sentiment={avg_polarity:+.3f} | risk={risk_score:.3f}")

df_mi_signals = pd.DataFrame(mi_signals_rows).sort_values("mi_score", ascending=False).reset_index(drop=True)
_path = GOLD_MI_DIR / "mi_signals.parquet"
df_mi_signals.to_parquet(_path, index=False, compression="snappy")
print(f"\n✅ mi_signals.parquet: {len(df_mi_signals)} FIIs")
print(df_mi_signals[["ticker","mentions","sentiment_avg","risk_score","opportunity_score","mi_score"]].to_string(index=False))


🧠 Calculando MI Signals por FII...
  HGLG11   →   73 docs | sentiment=+0.234 | risk=1.343
  KNRI11   →   83 docs | sentiment=+0.279 | risk=1.325
  XPML11   →   75 docs | sentiment=+0.276 | risk=1.160
  CSHG11   →   65 docs | sentiment=+0.230 | risk=1.462
  VISC11   →   68 docs | sentiment=+0.238 | risk=1.279
  BRCO11   →   71 docs | sentiment=+0.245 | risk=1.268
  BTLG11   →   73 docs | sentiment=+0.253 | risk=1.233
  MXRF11   →  102 docs | sentiment=+0.266 | risk=1.177
  HCRI11   →   90 docs | sentiment=+0.272 | risk=1.222
  ALZR11   →   85 docs | sentiment=+0.269 | risk=1.294
  BRCR11   →   67 docs | sentiment=+0.247 | risk=1.224
  HSML11   →   72 docs | sentiment=+0.214 | risk=1.278
  VGIP11   →   86 docs | sentiment=+0.260 | risk=1.233
  GGRC11   →   66 docs | sentiment=+0.250 | risk=1.227
  RBVA11   →   91 docs | sentiment=+0.173 | risk=1.418

✅ mi_signals.parquet: 15 FIIs
ticker  mentions  sentiment_avg  risk_score  opportunity_score  mi_score
HCRI11        90         0.2719    

## 📰 Seção 7 — MI Top Articles por FII

In [7]:

print("\n📰 Calculando MI Top Articles...")

top_articles_rows = []

for entity in FII_ENTITIES:
    ticker  = entity["ticker"]
    rel_dfs = [query_relevance(q, top_k=30) for q in entity["query_terms"]]
    df_rel  = pd.concat(rel_dfs, ignore_index=True)
    df_rel  = df_rel.groupby("article_id").agg(
        score_hybrid = ("score_hybrid","max"),
        score_bm25   = ("score_bm25",  "max"),
        score_tfidf  = ("score_tfidf", "max"),
    ).reset_index()

    df_top = df_rel.merge(
        df_silver[["article_id","title","url","source","source_type",
                   "published_dt","collected_at","polarity_score",
                   "sentiment_label","flag_dividendo","flag_risco",
                   "flag_crise","flag_oportunidade"]].drop_duplicates("article_id"),
        on="article_id", how="left"
    )

    # MI score por artigo
    df_top["mi_article_score"] = (
        0.5 * df_top["score_hybrid"].fillna(0) +
        0.3 * df_top["polarity_score"].fillna(0).abs() +
        0.2 * df_top["flag_dividendo"].fillna(False).astype(float)
    ).round(4)

    df_top = (
        df_top
        .sort_values("mi_article_score", ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

    for rank_i, row in df_top.iterrows():
        top_articles_rows.append({
            "ticker":           ticker,
            "rank":             rank_i + 1,
            "article_id":       row.get("article_id",""),
            "title":            row.get("title",""),
            "url":              row.get("url",""),
            "source":           row.get("source",""),
            "published_dt":     row.get("published_dt",""),
            "sentiment_label":  row.get("sentiment_label","neutro"),
            "polarity_score":   row.get("polarity_score", 0.0),
            "score_bm25":       row.get("score_bm25",  0.0),
            "score_tfidf":      row.get("score_tfidf", 0.0),
            "score_hybrid":     row.get("score_hybrid",0.0),
            "mi_article_score": row.get("mi_article_score",0.0),
            "flag_dividendo":   bool(row.get("flag_dividendo", False)),
            "flag_risco":       bool(row.get("flag_risco",     False)),
            "flag_crise":       bool(row.get("flag_crise",     False)),
        })

df_mi_top = pd.DataFrame(top_articles_rows)
_path = GOLD_MI_DIR / "mi_top_articles.parquet"
df_mi_top.to_parquet(_path, index=False, compression="snappy")
print(f"✅ mi_top_articles.parquet: {len(df_mi_top):,} linhas ({len(FII_ENTITIES)} FIIs × até 10 artigos)")


📰 Calculando MI Top Articles...
✅ mi_top_articles.parquet: 150 linhas (15 FIIs × até 10 artigos)


## 🔄 Seção 8 — Funil TOFU/MOFU/BOFU

In [8]:

print("\n🔄 Calculando Funil TOFU/MOFU/BOFU...")

funil_rows = []

for stage, queries in FUNIL_QUERIES.items():
    for q in queries:
        df_q = query_relevance(q, top_k=20)
        df_q = df_q.merge(
            df_silver[["article_id","source","source_type","sentiment_label",
                       "polarity_score","flag_dividendo","flag_risco"]].drop_duplicates("article_id"),
            on="article_id", how="left"
        )
        for _, row in df_q.iterrows():
            funil_rows.append({
                "stage":          stage,
                "query":          q,
                "article_id":     row.get("article_id",""),
                "source":         row.get("source",""),
                "source_type":    row.get("source_type",""),
                "score_hybrid":   row.get("score_hybrid",0.0),
                "sentiment_label":row.get("sentiment_label","neutro"),
                "polarity_score": row.get("polarity_score",0.0),
                "flag_dividendo": bool(row.get("flag_dividendo",False)),
                "flag_risco":     bool(row.get("flag_risco",False)),
            })

df_funil = pd.DataFrame(funil_rows)
_path = GOLD_MI_DIR / "mi_funnel.parquet"
df_funil.to_parquet(_path, index=False, compression="snappy")

print(f"✅ mi_funnel.parquet: {len(df_funil):,} registros")
print(f"\n   Resumo por estágio:")
_summ = (
    df_funil.groupby("stage")
    .agg(n_docs=("article_id","count"), avg_score=("score_hybrid","mean"),
         avg_polarity=("polarity_score","mean"))
    .round(4)
)
print(_summ.to_string())


🔄 Calculando Funil TOFU/MOFU/BOFU...
✅ mi_funnel.parquet: 300 registros

   Resumo por estágio:
       n_docs  avg_score  avg_polarity
stage                                 
BOFU      100     0.5328        0.3067
MOFU      100     0.4569        0.2752
TOFU      100     0.5108        0.4198


## ⚡ Seção 9 — Agregações Spark: Ranking de Fontes por MI Score

In [9]:

print("\n⚡ Spark: ranking de fontes por MI score médio...")

# Juntar top articles com sinais
_df_src_mi = df_mi_top.copy()
sdf_mi = spark.createDataFrame(_df_src_mi.fillna(0.0))

sdf_src_rank = (
    sdf_mi
    .groupBy("source")
    .agg(
        F.count("article_id").alias("n_articles"),
        F.avg("score_hybrid").alias("avg_relevance"),
        F.avg("polarity_score").alias("avg_polarity"),
        F.sum(F.col("flag_dividendo").cast("int")).alias("n_dividendo"),
        F.sum(F.col("flag_risco").cast("int")).alias("n_risco"),
    )
    .withColumn("source_mi_score",
        F.round(0.5 * F.col("avg_relevance") + 0.3 * F.abs("avg_polarity"), 4))
    .orderBy(F.col("source_mi_score").desc())
)

df_src_rank = sdf_src_rank.toPandas()
_path = GOLD_MI_DIR / "mi_source_ranking.parquet"
df_src_rank.to_parquet(_path, index=False, compression="snappy")
print(f"✅ mi_source_ranking.parquet: {len(df_src_rank)} fontes")
print(df_src_rank.to_string(index=False))


⚡ Spark: ranking de fontes por MI score médio...


✅ mi_source_ranking.parquet: 12 fontes
                source  n_articles  avg_relevance  avg_polarity  n_dividendo  n_risco  source_mi_score
      cnnbrasil.com.br           1       1.000000     -0.650000            0        0           0.6950
borainvestir.b3.com.br           3       0.897067      0.550000            3        0           0.6135
           fiis.com.br          11       0.815691      0.627273            8        0           0.5960
   investidor10.com.br          17       0.700024      0.613118           17       10           0.5339
   statusinvest.com.br          17       0.636406      0.691912           17        0           0.5258
       news.google.com           2       0.602150      0.733300            2        0           0.5211
  fundsexplorer.com.br          38       0.760000      0.464197           37       19           0.5193
      empiricus.com.br          31       0.828555      0.338119           27        8           0.5157
       seudinheiro.com           5

## 📋 Seção 10 — Resumo de Outputs

In [10]:

print("\n📋 OUTPUTS NB06 — data/gold/marketing_intelligence/")
print("=" * 65)
outputs = [
    ("mi_signals.parquet",        "Métricas por FII (15 entidades)"),
    ("mi_top_articles.parquet",   "Top 10 artigos por FII × score MI"),
    ("mi_funnel.parquet",         "Documentos por estágio TOFU/MOFU/BOFU"),
    ("mi_source_ranking.parquet", "Ranking de fontes por MI score"),
]
for fname, desc in outputs:
    p = GOLD_MI_DIR / fname
    if p.exists():
        sz  = p.stat().st_size // 1024
        df_ = pd.read_parquet(p)
        print(f"  ✅ {fname:<38} {sz:>5} KB | {len(df_):>5} linhas — {desc}")
    else:
        print(f"  ❌ {fname} — não gerado")

log_quality_check(logger, nb="NB06",
    fii_entities=len(FII_ENTITIES),
    mi_signals_rows=len(df_mi_signals),
    top_articles_rows=len(df_mi_top),
    funil_rows=len(df_funil))


📋 OUTPUTS NB06 — data/gold/marketing_intelligence/
  ✅ mi_signals.parquet                        11 KB |    15 linhas — Métricas por FII (15 entidades)
  ✅ mi_top_articles.parquet                   23 KB |   150 linhas — Top 10 artigos por FII × score MI
  ✅ mi_funnel.parquet                         15 KB |   300 linhas — Documentos por estágio TOFU/MOFU/BOFU
  ✅ mi_source_ranking.parquet                  5 KB |    12 linhas — Ranking de fontes por MI score
2026-06-14T15:52:08 | INFO     | fii_pipeline.ingestion | QUALITY_CHECK | nb=NB06 | fii_entities=15 | mi_signals_rows=15 | top_articles_rows=150 | funil_rows=300



## ✅ NB06 Completo

| Artefato | Colunas-chave | Consumido por |
|----------|---------------|---------------|
| `mi_signals.parquet` | ticker, sentiment_avg, risk_score, opportunity_score, mi_score | NB07 |
| `mi_top_articles.parquet` | ticker, rank, title, score_hybrid, sentiment_label | NB07 |
| `mi_funnel.parquet` | stage, query, source, score_hybrid | NB07 |
| `mi_source_ranking.parquet` | source, avg_relevance, source_mi_score | NB07 |

▶️ **Próximo:** `07_dashboard_dataset.ipynb`